# 02 · FlashSAC 실습: entropy, reward scaling, noise repetition

목표: 논문의 수식 (6), (7)과 시간 상관 탐색을 NumPy로 구현합니다. 실제 SAC agent나 robot simulator는 사용하지 않는 toy reproduction입니다.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(11)

## 1. 통합 entropy target

$$\bar{\mathcal H}=\frac{1}{2}|\mathcal A|\log(2\pi e\sigma_{\mathrm{tgt}}^2)$$

목표 action 표준편차를 고정하면 target entropy가 action dimension에 선형으로 변합니다.

In [ ]:
def entropy_target(action_dim, sigma_target=0.15):
    return 0.5 * action_dim * np.log(2 * np.pi * np.e * sigma_target**2)

dimensions = np.array([2, 6, 12, 29])
targets = np.array([entropy_target(dim) for dim in dimensions])
for dim, target in zip(dimensions, targets):
    print(f'action_dim={dim:2d}, target={target: .4f}')
assert np.allclose(targets / dimensions, targets[0] / dimensions[0])

## 2. Adaptive reward scaling

$$\bar r_t=\frac{r_t}{\max(\sqrt{\sigma^2_{t,G}+\epsilon},G_{t,\max}/G_{\max})}$$

아래 코드는 논문의 직관을 보여 주는 단순화 구현입니다. 공식 코드의 running statistics 갱신과 동일하다고 주장하지 않습니다.

In [ ]:
def adaptive_scale(rewards, return_variance, running_max, support_max, eps=1e-8):
    denominator = max(np.sqrt(return_variance + eps), running_max / support_max)
    return np.asarray(rewards) / denominator, denominator

rewards = np.array([-20.0, -2.0, 0.0, 5.0, 30.0])
scaled, denominator = adaptive_scale(
    rewards, return_variance=64.0, running_max=120.0, support_max=10.0
)
print('denominator:', denominator)
print('scaled rewards:', scaled)
assert denominator == 12.0
assert np.max(np.abs(scaled)) < np.max(np.abs(rewards))

## 3. Noise repetition

독립 noise는 매 step 방향이 바뀝니다. Noise repetition은 Zeta 분포에서 반복 길이를 뽑아 같은 noise를 여러 step 유지합니다. 그 결과 lag-1 autocorrelation이 커지고 더 일관된 탐색 구간이 생깁니다.

In [ ]:
def repeated_noise(length, exponent=2.0):
    values = []
    while len(values) < length:
        repeat = int(rng.zipf(exponent))
        noise = float(rng.normal())
        values.extend([noise] * repeat)
    return np.array(values[:length])

def lag1_correlation(values):
    return np.corrcoef(values[:-1], values[1:])[0, 1]

length = 2000
iid = rng.normal(size=length)
repeated = repeated_noise(length)
print(f'i.i.d. lag-1 correlation: {lag1_correlation(iid):.3f}')
print(f'repeated lag-1 correlation: {lag1_correlation(repeated):.3f}')
assert lag1_correlation(repeated) > lag1_correlation(iid) + 0.2

fig, axes = plt.subplots(2, 1, figsize=(11, 5), sharex=True)
axes[0].plot(iid[:150], lw=1)
axes[0].set_title('Independent action noise')
axes[1].plot(repeated[:150], lw=1, color='tab:green')
axes[1].set_title('Zeta-length noise repetition')
axes[1].set_xlabel('step')
plt.tight_layout();

## 응용 질문

1. Zeta exponent를 1.5, 2.0, 3.0으로 바꾸면 평균 반복 길이와 autocorrelation이 어떻게 변합니까?
2. action dimension이 커질 때 같은 $\sigma_{\mathrm{tgt}}$의 target entropy가 왜 선형으로 변해야 합니까?
3. reward scaling의 denominator를 너무 크게 잡으면 critic 학습 신호에 어떤 문제가 생깁니까?

다음 notebook에서는 bootstrap target이 있는 toy critic에서 weight projection이 norm 폭주를 어떻게 바꾸는지 관찰합니다.